# 랭체인으로 RAG 시작하기

제작 : 박광석(모두의연구소, https://www.linkedin.com/in/andkspark)

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다



```
# 코드로 형식 지정됨
```

### Step 0 : 설치와 준비  
Langchain 설치 및 OpenAI API 키를 등록하도록 합니다.  

In [1]:
# ✅ Step 0: 전체 환경 설정 셀
# 이 셀만 먼저 실행하면, 아래 셀들은 그대로 실행해도 됩니다.

# 1) 필수 패키지 설치
!pip install -q \
  requests==2.32.5 \
  langchain langchain-openai langchain-community langchain-core langchain-classic \
  chromadb pypdf tiktoken sentence-transformers \
  langchain-huggingface langchain-chroma langchain-text-splitters \
  pdf2image docx2txt pdfminer.six unstructured

# 2) 기본 파이썬 모듈 및 환경 변수 설정
import os
from dotenv import load_dotenv

# 노트북과 같은 폴더 기준 (Jupyter 실행 시 cwd가 프로젝트 폴더)
BASE_DIR = os.getcwd()

# .env에서 OPENAI_API_KEY 로드
load_dotenv(os.path.join(BASE_DIR, ".env"))
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 3) LangChain / RAG에 필요한 공통 임포트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

# 4) 토큰 길이 계산용 tiktoken 설정
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text: str) -> int:
    tokens = tokenizer.encode(text)
    return len(tokens)

print("환경 설정 완료: 패키지 설치 및 공통 임포트, OPENAI_API_KEY 로드")

USER_AGENT environment variable not set, consider setting it to identify your requests.


환경 설정 완료: 패키지 설치 및 공통 임포트, OPENAI_API_KEY 로드


In [2]:
!pip install pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

In [3]:
! curl ipinfo.io

{
  "ip": "112.169.232.130",
  "city": "Seoul",
  "region": "Seoul",
  "country": "KR",
  "loc": "37.5660,126.9784",
  "org": "AS4766 Korea Telecom",
  "postal": "03141",
  "timezone": "Asia/Seoul",
  "readme": "https://ipinfo.io/missingauth"
}

In [4]:
# 로컬 환경: 노트북과 같은 폴더 기준 (Jupyter 실행 시 cwd가 프로젝트 폴더)
# import os
BASE_DIR = os.getcwd()

In [5]:
# from dotenv import load_dotenv
# import os
# 노트북과 같은 폴더의 .env에서 API 키 로드
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [ ]:
# !pip install requests==2.32.5
# !pip install -U -q langchain langchain-openai
# !pip install -U -q langchain-community langchain-core
# !pip install -q langchain-classic
# !pip install chromadb pypdf tiktoken sentence-transformers langchain-chroma



In [ ]:
# import os
# os.environ['GEMINA_API_KEY'] = YOUR_API_KEY

In [6]:
#랭체인의 OPENAI Api를 사용합니다
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.0)

In [ ]:
# !pip install pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, Demian.pdf를 프로젝트 폴더에 넣어주세요. (없다면 다른 PDF로 경로만 수정)

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(os.path.join(BASE_DIR, "Demian.pdf"))
pages = loader.load_and_split()

In [8]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/Users/jungsunpark/내 드라이브(darkaruna78@gmail.com)/modulab/Data_Quset/09 RAG/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [9]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [10]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, titanic.csv를 프로젝트 폴더에 넣어주세요. (이미 포함됨)

In [11]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(os.path.join(BASE_DIR, "titanic.csv"))

data = loader.load()

In [12]:
data[:3]

[Document(metadata={'source': '/Users/jungsunpark/내 드라이브(darkaruna78@gmail.com)/modulab/Data_Quset/09 RAG/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/Users/jungsunpark/내 드라이브(darkaruna78@gmail.com)/modulab/Data_Quset/09 RAG/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/Users/jungsunpark/내 드라이브(darkaruna78@gmail.com)/modulab/Data_Quset/09 RAG/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [13]:
from langchain_community.document_loaders import WebBaseLoader

In [14]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

print(documents[0].page_content)





































모두의연구소 ‘AI학교 아이펠’, ICLR 2024 워크숍 혁신 기술 논문 채택 < 일반 < 기업 < 기사본문 - IT조선










 






















































주요서비스 바로가기
본문 바로가기
매체정보 바로가기
로그인 바로가기
기사검색 바로가기
전체서비스 바로가기




























 






전체메뉴



기사검색








기사검색


검색

닫기














기사검색









기사검색


검색

닫기











로그인




facebook

post
youtube



UPDATED. 2026-03-17 17:04 (화) 




기사검색









기업

일반
모바일·가전
방송·통신
반도체·디스플레이
SW·보안
중공업·에너지
소부장·스타트업
유통·쇼핑
프롭테크·부동산



모빌리티

자동차·모빌리티
로봇·드론·항공
방산



게임·콘텐츠

게임·인터넷
메타버스·VR
키덜트
미디어·엔터



과학·헬스

과학·우주
의학·정책
제약·바이오



파이낸스

금융
증권
핀테크·블록체인



칼럼·인터뷰

칼럼
기고
인터뷰



알림

알립니다
인사
부음
새로나왔어요



컴퓨팅·AI

일반
코딩
에듀테크
테크리포트



GLOBAL
기획·연재
속보
백과사전











속보




기업


일반


모바일·가전


방송·통신


반도체·디스플레이


SW·보안


중공업·에너지


소부장·스타트업


유통·쇼핑


프롭테크·부동산




모빌리티


자동차·모빌리티


로봇·드론·항공


방산




게임·콘텐츠


게임·인터넷


메타버스·VR


키덜트


미디어·엔터




과학·헬스


과학·우주


의학·정책


제약·바이오




파이낸스


금융


증권


핀테크·블록체인




칼럼·인터뷰


칼럼


기고


인터뷰



주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [15]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [16]:
with open(os.path.join(BASE_DIR, "state_of_the_union.txt")) as f:
    text = f.read()

In [17]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [18]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [19]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [ ]:
# !pip install tiktoken

In [20]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [21]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI의 text-embedding-3-small 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

공식 문서는 아래 링크에서 확인할 수 있습니다.
https://ai.google.dev/docs/embeddings_guide?hl=ko

In [ ]:
# OpenAI는 langchain-openai에 포함 (Step 0에서 이미 설치)

OpenAI의 임베딩 모델(text-embedding-3-small 등)을 사용합니다.

In [ ]:
# OpenAI API 키는 .env에서 이미 로드됨 (OPENAI_API_KEY)

In [ ]:
# OpenAI 임베딩 모델: text-embedding-3-small, text-embedding-3-large

OpenAI의 text-embedding-3-small 모델을 사용합니다.  

https://platform.openai.com/docs/guides/embeddings

In [22]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

OpenAI embedding은 전 세계에서 사용 가능합니다.  
API 키가 .env 파일에 OPENAI_API_KEY로 설정되어 있는지 확인하세요.


In [ ]:
# !curl ipinfo.io

In [ ]:

# !pip install -U langchain-huggingface sentence-transformers

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=50,
    length_function=tiktoken_len,
)
texts = text_splitter.split_documents(pages)

In [26]:
# import os
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

# from langchain.embeddings import HuggingFaceEmbeddings
# from langchain.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma  # ← 여기!


embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
docsearch = Chroma.from_documents(texts, embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model 변수에 google의 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [27]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [28]:
print(embeddings[1])

[-0.043598722666502, 0.0467681959271431, -0.026038112118840218, -0.001370531041175127, 0.060555584728717804, 0.05017252638936043, 0.11730730533599854, -0.027721047401428223, 0.009279531426727772, 0.07751472294330597, 0.0015128167578950524, -0.09486881643533707, 0.0018309853039681911, 0.010358046740293503, 0.02260894514620304, 0.043930016458034515, -0.013756045140326023, -0.020579224452376366, -0.032713908702135086, -0.025614814832806587, 0.04354719817638397, 0.08289314806461334, -0.05014248564839363, -0.04123268648982048, -0.05022474750876427, 0.04015064984560013, 0.057477738708257675, 0.012200244702398777, 0.028081491589546204, -0.060744479298591614, -0.06629892438650131, 0.0234212763607502, 0.048590946942567825, -0.015047362074255943, -0.020352844148874283, -0.0033349941950291395, 0.03370741009712219, -0.10650119930505753, 0.034283172339200974, 0.06269638985395432, 0.03697559982538223, 0.0399332195520401, 0.06028096377849579, -0.05295759439468384, -0.02507840469479561, 0.066467292606

In [29]:
len(embeddings[1])

384

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [30]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [31]:
query = ["this is red fruit"]

In [32]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7799127588507688
0.6021169766774179
0.5388710261186725


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [ ]:
# !pip install chromadb

In [ ]:
# !pip install langchain-chroma

In [ ]:
# from langchain_community.vectorstores import Chroma

In [ ]:
# from langchain_chroma import Chroma

In [ ]:
# !pip install --upgrade opentelemetry-api
# !pip install --upgrade opentelemetry-sdk

In [ ]:
# from langchain_chroma import Chroma

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [33]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader(os.path.join(BASE_DIR, "Demian.pdf"))
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [34]:
!pip show chromadb

Name: chromadb
Version: 1.5.5
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /opt/homebrew/lib/python3.11/site-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: langchain-chroma


Chroma에 임베딩 시킵니다  

In [35]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [36]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [37]:
print(docs[0].page_content)

DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face somehow expressed my life, it was my inner self, 
my fate or my daimon. That was how

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [ ]:
# !pip install --upgrade langchain

In [ ]:
# # from langchain_core.prompts import ChatPromptTemplate
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from langchain_classic.chains import create_retrieval_chain


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [38]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [ ]:
# qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
#                                  retriever=db.as_retriever(
#                                      search_type="mmr",
#                                      search_kwargs={"k": 3, "fetch_k" : 10}),
#                                  return_source_documents=True)


# 1) Retriever (기존 docsearch 그대로 사용)
retriever = docsearch.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10},
)

# 2) LLM에 줄 프롬프트
prompt = ChatPromptTemplate.from_template(
    """
당신은 주어진 문서들을 바탕으로 질문에 답하는 QA 어시스턴트입니다.

<context>
{context}
</context>

질문: {input}

위 context에 없는 내용은 모른다고 답하세요.
    """.strip()
)

# 3) 문서들을 LLM에 넣어 답변을 만드는 체인
document_chain = create_stuff_documents_chain(llm, prompt)

# 4) 최종 RAG 체인
qa_chain = create_retrieval_chain(retriever, document_chain)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [40]:
query = "how demian looks like"
result = qa_chain.invoke({"input": query})

마크다운 형식으로 출력해봅니다

In [41]:
from IPython.display import Markdown, display
display(Markdown(result["answer"]))

주어진 문서에서는 Demian의 외모에 대한 구체적인 묘사는 없지만, 그의 얼굴이 종이 위의 그림과 비슷하다고 언급되고 있습니다. 또한, Demian의 특징은 "밝고, 똑똑하며, 비범한 결단력을 가진 얼굴"로 묘사되며, 그는 다른 아이들과는 다르게 자신만의 개성을 지닌 인물로 보입니다. 그러나 구체적인 외모에 대한 자세한 설명은 제공되지 않았습니다.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [42]:

llm2 = ChatOpenAI(model = "gpt-4o-mini", temperature=0.0)
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


"Demian" is a novel by Hermann Hesse, published in 1919. The story follows a young man named Emil Sinclair as he navigates his journey of self-discovery and explores themes of duality, identity, and the search for meaning. 

In terms of physical appearance, the character Demian, who plays a significant role in Sinclair's life, is often described as having an intense and charismatic presence. He is depicted as confident, with a strong personality that draws others to him. However, specific details about his appearance can vary based on individual interpretations and adaptations of the novel.

If you are referring to a specific adaptation of "Demian," such as a film or graphic novel, please provide more details, and I can give you a more tailored description!

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
다음은 Lagnchain으로 구현된 Question-Answer RAG 완성 예제입니다  


In [ ]:
# 로컬 환경: 위쪽 셀에서 이미 BASE_DIR, load_dotenv 설정됨

In [ ]:
# .env에서 API 키 로드 (Step 0에서 이미 실행했다면 생략 가능)

필요한 라이브러리를 모두 다운받습니다  

In [ ]:
# !pip install -q langchain langchain-openai chromadb pypdf sentence_transformers tiktoken

In [ ]:
# !pip install -U -q langchain-community langchain-core

In [ ]:
# from dotenv import load_dotenv
# import os
# BASE_DIR = os.getcwd()  # 완성 예제만 단독 실행 시
# load_dotenv(os.path.join(BASE_DIR, ".env"))
# os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [ ]:
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from langchain_classic.chains import create_retrieval_chain
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain.document_loaders import PyPDFLoader
# from langchain.vectorstores import Chroma

Text splitter 사용을 위한 준비입니다

In [43]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)

    return len(tokens)

### Step 1 Document loader

In [44]:
loader = PyPDFLoader(os.path.join(BASE_DIR, "Demian.pdf"))
pages = loader.load_and_split()



### Step 2 Text splitters

In [45]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=5000, chunk_overlap=50, length_function = tiktoken_len)
texts = text_splitter.split_documents(pages)

### Step 3 Vector Empeddings

In [ ]:
# from langchain.embeddings import HuggingFaceEmbeddings(과거))
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
docsearch = Chroma.from_documents(texts, embedding_model)

### Step 4 Retrievers

In [49]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_openai import ChatOpenAI
# QA

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

In [50]:
# 1) Retriever
retriever = docsearch.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10},
)

# 2) 프롬프트
prompt = ChatPromptTemplate.from_template("""
당신은 주어진 문서들을 바탕으로 질문에 답하는 QA 어시스턴트입니다.

<context>
{context}
</context>

질문: {input}

위 context에 없는 내용은 모른다고 답하세요.
""".strip())

# 3) 문서 체인 + RAG 체인
document_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(retriever, document_chain)

### Question Answering

In [51]:
query = "how demian looks like"
result = qa_chain.invoke({"input": query})

In [52]:
from IPython.display import Markdown, display
display(Markdown(result["answer"]))

주어진 문서에서는 Demian의 외모에 대한 구체적인 묘사는 없지만, 그의 얼굴이 기억 속에서 비슷하다고 언급되고 있습니다. 또한, Demian의 특징은 "밝고, 똑똑하며, 비범한 결단력을 가진 얼굴"로 묘사되며, 그는 다른 아이들과는 다르게 자신만의 개성을 지닌 인물로 보입니다. 그러나 구체적인 외모에 대한 자세한 설명은 제공되지 않았습니다.